# IBClient Output Probe

Interactive probe for `IBClient` request outputs in Jupyter.

Run cells top-to-bottom.

In [5]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0]
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from ib_insync import IB, Forex, LimitOrder, util
from services.ib_client import IBClient

# Required in Jupyter to avoid: RuntimeError("This event loop is already running")
util.startLoop()

print(f"Project root: {ROOT}")

Project root: C:\Users\Valérian Darmenté\Documents\Python Project


In [14]:
# Configuration
HOST = "127.0.0.1"
PORT = 4002
CLIENT_ID = 22
READONLY = True
SYMBOL = "EURUSD"
WAIT_SECONDS = 1.0
POLL_SECONDS = 20
POLL_INTERVAL = 1.0
PROBE_ORDERS = True  # Set True only when you intentionally want order probes.

In [7]:
def show(name, value):
    print(f"{name:<40} -> {value}")

In [19]:
# Setup
CLIENT_ID = 26 # Change ID IF who Cant Connect
ib = IB()
client = IBClient(
    ib=ib,
    host=HOST,
    port=PORT,
    client_id=CLIENT_ID,
    readonly=READONLY,
)
contract = Forex(SYMBOL)

show("connect_and_prepare()", "calling...")
client.connect_and_prepare()
show("is_connected()", client.is_connected())

connect_and_prepare()                    -> calling...
is_connected()                           -> True


In [11]:
# Core method outputs
client.process_messages()
show("process_messages()", "done")
show("_is_valid_price(None)", client._is_valid_price(None))
show("_is_valid_price(1.23)", client._is_valid_price(1.23))
show("get_latest_bid_ask()", client.get_latest_bid_ask())
show("get_portfolio_snapshot()", client.get_portfolio_snapshot())
show("get_connection_mode()", client.get_connection_mode())
show("get_environment()", client.get_environment())
show("get_account()", client.get_account())
show("supports_server_time()", client.supports_server_time())
show("get_server_time_and_latency()", client.get_server_time_and_latency())
show("get_status_snapshot()", client.get_status_snapshot())

process_messages()                       -> done
_is_valid_price(None)                    -> False
_is_valid_price(1.23)                    -> True
get_latest_bid_ask()                     -> (-1.0, -1.0)
get_portfolio_snapshot()                 -> ([AccountValue(account='DUM194375', tag='AccountType', value='INDIVIDUAL', currency='', modelCode=''), AccountValue(account='DUM194375', tag='Cushion', value='1', currency='', modelCode=''), AccountValue(account='DUM194375', tag='LookAheadNextChange', value='0', currency='', modelCode=''), AccountValue(account='DUM194375', tag='AccruedCash', value='483.51', currency='EUR', modelCode=''), AccountValue(account='DUM194375', tag='AvailableFunds', value='1010701.05', currency='EUR', modelCode=''), AccountValue(account='DUM194375', tag='BuyingPower', value='6738007.00', currency='EUR', modelCode=''), AccountValue(account='DUM194375', tag='EquityWithLoanValue', value='1010701.05', currency='EUR', modelCode=''), AccountValue(account='DUM194375', tag

In [15]:
# Market/account request outputs
qualified = client.qualify_contract(contract)
show("qualify_contract()", qualified)
contract_for_queries = qualified or contract

show("get_market_snapshot()", client.get_market_snapshot(contract_for_queries, wait_seconds=WAIT_SECONDS))
bars = client.get_historical_bars(
    contract_for_queries,
    duration="1 D",
    bar_size="1 min",
    what_to_show="MIDPOINT",
    use_rth=False,
)
show("get_historical_bars()", f"{len(bars)} bars")
show("get_head_timestamp()", client.get_head_timestamp(contract_for_queries, what_to_show="MIDPOINT"))
show("get_contract_details()", client.get_contract_details(contract_for_queries))
show("get_account_values()", client.get_account_values())
show("request_account_summary()", client.request_account_summary(group_name="All", tags="NetLiquidation,AvailableFunds"))
stream_ticker = client.request_market_data(
    contract_for_queries,
    generic_tick_list="",
    snapshot=False,
    regulatory_snapshot=False,
)
show("request_market_data()", stream_ticker)
show("cancel_market_data()", client.cancel_market_data(contract_for_queries))
show("get_open_orders_snapshot()", client.get_open_orders_snapshot())
show("request_all_open_orders()", client.request_all_open_orders())
show("request_completed_orders()", client.request_completed_orders(api_only=False))
show("get_open_trades_snapshot()", client.get_open_trades_snapshot())
show("get_fills_snapshot()", client.get_fills_snapshot())
show("get_executions_snapshot()", client.get_executions_snapshot())

account = client.get_account()
pnl = client.get_pnl_snapshot(account=account)
show("get_pnl_snapshot()", pnl)
con_id = getattr(contract_for_queries, "conId", 0) or 0
pnl_single = client.get_pnl_single_snapshot(account=account, con_id=con_id)
show("get_pnl_single_snapshot()", pnl_single)
show("get_market_depth_snapshot()", client.get_market_depth_snapshot(contract_for_queries, num_rows=5))

qualify_contract()                       -> Forex('EURUSD', conId=12087792, exchange='IDEALPRO', localSymbol='EUR.USD', tradingClass='EUR.USD')
get_market_snapshot()                    -> {'bid': -1.0, 'ask': -1.0, 'last': nan, 'close': 1.1871, 'volume': 0.0, 'time': datetime.datetime(2026, 2, 14, 15, 41, 0, 885961, tzinfo=datetime.timezone.utc)}
get_historical_bars()                    -> 1425 bars
get_head_timestamp()                     -> 2005-03-08 22:15:00
get_contract_details()                   -> [ContractDetails(contract=Contract(secType='CASH', conId=12087792, symbol='EUR', exchange='IDEALPRO', currency='USD', localSymbol='EUR.USD', tradingClass='EUR.USD'), marketName='EUR.USD', minTick=5e-05, orderTypes='ACTIVETIM,AD,ADJUST,ALERT,ALGO,ALLOC,AVGCOST,BASKET,BENCHPX,CASHQTY,COND,CONDORDER,DAY,DEACT,DEACTDIS,DEACTEOD,GAT,GTC,GTD,GTT,HID,IOC,LIT,LMT,MIT,MKT,NONALGO,OCA,REL,RELPCTOFS,SCALE,SCALERST,STP,STPLMT,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF', validExchanges='IDEALPRO', pr

In [16]:
# Optional order probes
if PROBE_ORDERS:
    probe_order = LimitOrder("BUY", 1000, 0.5)
    show("what_if_order()", client.what_if_order(contract_for_queries, probe_order))
    if READONLY:
        show("place_order()", "skipped (READONLY=True)")
        show("replace_order()", "skipped (READONLY=True)")
        show("cancel_order()", "skipped (READONLY=True)")
    else:
        submitted_trade = client.place_order(contract_for_queries, probe_order)
        show("place_order()", submitted_trade)
        replaced_trade = client.replace_order(contract_for_queries, probe_order)
        show("replace_order()", replaced_trade)
        cancel_target = submitted_trade if submitted_trade is not None else probe_order
        show("cancel_order()", client.cancel_order(cancel_target))
else:
    show("order_probe", "skipped (set PROBE_ORDERS=True to enable)")

what_if_order()                          -> []
place_order()                            -> skipped (READONLY=True)
replace_order()                          -> skipped (READONLY=True)
cancel_order()                           -> skipped (READONLY=True)


In [17]:
# Live polling
print("\nLive polling:")
end = time.time() + POLL_SECONDS
while time.time() < end:
    client.process_messages()
    show("get_latest_bid_ask()", client.get_latest_bid_ask())
    time.sleep(POLL_INTERVAL)


Live polling:
get_latest_bid_ask()                     -> (-1.0, -1.0)
get_latest_bid_ask()                     -> (-1.0, -1.0)
get_latest_bid_ask()                     -> (-1.0, -1.0)
get_latest_bid_ask()                     -> (-1.0, -1.0)
get_latest_bid_ask()                     -> (-1.0, -1.0)


KeyboardInterrupt: 

In [18]:
# Cleanup (run when done)
client.cancel_account_summary()
if ib.isConnected():
    ib.disconnect()
show("disconnect", "done")

disconnect                               -> done
